In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
plt.style.use('ggplot')
sns.set_palette('tab10')

## 1. Initial Inspection

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')
sub = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sub shape :", sub.shape)

print("\nTarget distribution:")
print(train["Heart Disease"].value_counts(normalize=True).round(3))

train_ids = train["id"].copy()
test_ids  = test["id"].copy()

train = train.drop(columns="id")
test  = test.drop(columns="id")

print("\nMissing values in train:\n", train.isna().sum().sum())
print("Missing values in test :", test.isna().sum().sum())

target = "Heart Disease"

categorical_features = [
    'Sex',              
    'Chest pain type',  
    'FBS over 120',     
    'EKG results',      
    'Exercise angina',  
    'Slope of ST',      
    'Thallium'          
]

numerical_features = [
    'Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression',
    'Number of vessels fluro'
]

for col in categorical_features:
    train[col] = train[col].astype('category')
    test[col]  = test[col].astype('category')

print("Categorical features:", categorical_features)
print("Numerical features  :", numerical_features)

## 2. Feature Engineering

In [ ]:
# risk-related ratios & interactions
train['Age_Chol_ratio']     = train['Age'] / (train['Cholesterol'] + 1e-5)
test['Age_Chol_ratio']      = test['Age']  / (test['Cholesterol'] + 1e-5)

train['BP_age']             = train['BP'] * train['Age']
test['BP_age']              = test['BP']  * test['Age']

train['ST_maxHR']           = train['ST depression'] / (train['Max HR'] + 1e-5)
test['ST_maxHR']            = test['ST depression']  / (test['Max HR'] + 1e-5)

# binary risk flags 
train['High_BP']            = (train['BP'] >= 140).astype(int)
test['High_BP']             = (test['BP'] >= 140).astype(int)

train['High_Chol']          = (train['Cholesterol'] >= 240).astype(int)
test['High_Chol']           = (test['Cholesterol'] >= 240).astype(int)

train['Low_MaxHR_age']      = (train['Max HR'] < 120 + 0.5 * (220 - train['Age'])).astype(int)  # rough expected max HR rule
test['Low_MaxHR_age']       = (test['Max HR']  < 120 + 0.5 * (220 - test['Age'])).astype(int)

# aggregate risk score 
risk_features = ['High_BP', 'High_Chol', 'Exercise angina', 'FBS over 120', 'Low_MaxHR_age']
train['Risk_score_simple']  = train[risk_features].sum(axis=1)
test['Risk_score_simple']   = test[risk_features].sum(axis=1)

print("New features added. New train shape:", train.shape)

## 3. Training & Evaluation

In [ ]:
X = train.drop(columns=[target])
y = train[target]

cat_features = categorical_features.copy()  

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
oof = np.zeros(len(train))
preds = np.zeros(len(test))

for fold, (idx_tr, idx_val) in enumerate(cv.split(X, y)):
    X_tr, y_tr = X.iloc[idx_tr], y.iloc[idx_tr]
    X_val, y_val = X.iloc[idx_val], y.iloc[idx_val]
    
    model = cb.CatBoostClassifier(
        iterations       = 12000,
        learning_rate    = 0.035,
        depth            = 7,
        l2_leaf_reg      = 2.5,
        border_count     = 128,
        bootstrap_type   = 'Bernoulli',
        subsample        = 0.85,
        eval_metric      = 'AUC',
        random_seed      = 42 + fold,
        verbose          = 600,
        early_stopping_rounds = 600,
        allow_writing_files = False
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set = (X_val, y_val),
        cat_features = cat_features,
        use_best_model = True
    )
    
    val_pred = model.predict_proba(X_val)[:, 1]
    oof[idx_val] = val_pred
    fold_auc = roc_auc_score(y_val, val_pred)
    scores.append(fold_auc)
    
    test_pred = model.predict_proba(test)[:, 1]
    preds += test_pred / cv.n_splits
    
    print(f"Fold {fold+1} AUC: {fold_auc:.5f}  | best iter: {model.best_iteration_}")

print(f"\nCV AUC: {np.mean(scores):.5f} ± {np.std(scores):.5f}")

In [ ]:
sub['Heart Disease'] = test_pred
sub.to_csv('submission_catboost_fe_v1.csv', index=False)
print("Submission saved.")
print(sub.head())